# Practical Decision Trees with Scikit-Learn

Welcome to this practical guide on Decision Trees! In this notebook, we will explore how to build, evaluate, and visualize decision trees for both classification and regression tasks using `scikit-learn`.

## 1. Introduction & Theory

Decision Trees are versatile machine learning algorithms that can perform both classification and regression tasks. They work by splitting the data into subsets based on the value of input features, creating a tree-like model of decisions.

**Key Concepts:**
- **Root Node:** The top node representing the entire dataset.
- **Splitting:** Dividing a node into two or more sub-nodes.
- **Decision Node:** A node that splits into further sub-nodes.
- **Leaf/Terminal Node:** Nodes that do not split (final prediction).
- **Pruning:** Removing sub-nodes to prevent overfitting.

**Splitting Criteria:**
- **Gini Impurity:** Measures the frequency at which any element of the dataset will be mislabeled when it is randomly labeled.
- **Entropy / Information Gain:** Measures the impurity or randomness in the data.

## 2. Data Preparation

Let's start by importing the necessary libraries and loading a classic dataset: the **Breast Cancer Wisconsin (Diagnostic) dataset** for classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn modules
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load classification dataset
cancer_data = load_breast_cancer()
X_class = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y_class = cancer_data.target

print(f"Dataset shape: {X_class.shape}")
print(f"Classes: {cancer_data.target_names}")

# Train-test split (80% train, 20% test)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(f"Training samples: {X_train_c.shape[0]}")
print(f"Testing samples: {X_test_c.shape[0]}")

## 3. Building a Classification Tree

We will use `DecisionTreeClassifier` from `scikit-learn` to build our model.

In [ ]:
# Initialize the classifier
# We'll start with default parameters (which often leads to overfitting)
dt_classifier = DecisionTreeClassifier(random_state=42)

# Train the model
dt_classifier.fit(X_train_c, y_train_c)

# Make predictions
y_pred_train_c = dt_classifier.predict(X_train_c)
y_pred_test_c = dt_classifier.predict(X_test_c)

print("Model trained successfully!")

## 4. Model Evaluation

Let's evaluate our model using accuracy, a confusion matrix, and a classification report.

In [ ]:
# Calculate accuracy
train_acc = accuracy_score(y_train_c, y_pred_train_c)
test_acc = accuracy_score(y_test_c, y_pred_test_c)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc:.4f}")
print("Notice the perfect training accuracy? This is a classic sign of overfitting!")

# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_test_c)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cancer_data.target_names, yticklabels=cancer_data.target_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_test_c, target_names=cancer_data.target_names))

## 5. Visualizing the Decision Tree

One of the biggest advantages of decision trees is their interpretability. We can visualize the tree structure to understand how it makes decisions.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(dt_classifier, 
          feature_names=cancer_data.feature_names,  
          class_names=cancer_data.target_names,
          filled=True, 
          rounded=True, 
          max_depth=3) # Limiting depth for better visibility
plt.title("Decision Tree Structure (Truncated to depth 3)")
plt.show()

## 6. Hyperparameter Tuning to Prevent Overfitting

Decision trees are prone to overfitting (creating overly complex trees that don't generalize well). We can control this using hyperparameters like `max_depth`, `min_samples_split`, and `min_samples_leaf`.

In [ ]:
# Define parameter grid
param_grid = {
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Initialize Grid Search
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), 
                           param_grid, 
                           cv=5, 
                           scoring='accuracy',
                           n_jobs=-1)

# Fit Grid Search
grid_search.fit(X_train_c, y_train_c)

print(f"Best Parameters: {grid_search.best_params_}")

# Evaluate best model
best_dt = grid_search.best_estimator_
y_pred_best = best_dt.predict(X_test_c)

print(f"\nNew Testing Accuracy: {accuracy_score(y_test_c, y_pred_best):.4f}")

## 7. Feature Importance

Decision trees can also tell us which features are most important for making predictions.

In [ ]:
# Get feature importances from the best model
importances = best_dt.feature_importances_

# Create a DataFrame for visualization
feat_imp_df = pd.DataFrame({
    'Feature': cancer_data.feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df.head(10), palette='viridis')
plt.title('Top 10 Most Important Features')
plt.tight_layout()
plt.show()

## 8. Regression Trees

Decision trees can also be used for regression tasks (predicting continuous values). Let's use the **California Housing dataset** for this.

In [ ]:
# Load regression dataset
housing = fetch_california_housing()
X_reg = pd.DataFrame(housing.data, columns=housing.feature_names)
y_reg = housing.target

# Train-test split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Initialize and train Regression Tree
# We set max_depth to prevent massive overfitting
dt_regressor = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_regressor.fit(X_train_r, y_train_r)

# Predict and evaluate
y_pred_r = dt_regressor.predict(X_test_r)
mse = mean_squared_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")

# Visualize predictions vs actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, y_pred_r, alpha=0.3)
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Regression Tree: Actual vs Predicted')
plt.show()